In [17]:
!pip install openai erniebot edge-tts

Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [edge-tts]


In [19]:
# ============================================================
# AI Creative Story Studio - Paddle AI Studio (Final)
# All messages in English to avoid encoding issues.
# ============================================================

import os
import asyncio
from datetime import datetime
from openai import OpenAI

# ==================== CONFIGURATION ====================
ACCESS_TOKEN = "a77cc4be4e1b4c726ae5a6195312183be80687e5"   # Replace with your actual token
BASE_URL = "https://aistudio.baidu.com/llm/lmapi/v3"
LLM_MODEL = "deepseek-v3"           # Use deepseek-v3 (well-supported)
OUTPUT_DIR = "outputs"
IMAGE_MODEL = "ernie-vilg-v2"
IMAGE_SIZE = "512x512"
TTS_VOICE = "zh-CN-XiaoxiaoNeural"
# ========================================================

client = OpenAI(api_key=ACCESS_TOKEN, base_url=BASE_URL)

# ---------- Story Generation ----------
def generate_story(topic, max_length=500):
    prompt = f"""请根据以下主题创作一个富有创意和画面感的短篇故事，字数控制在{max_length}字左右：

主题：{topic}

要求：
1. 故事结构完整，有开头、发展和结尾
2. 语言生动，富有想象力
3. 适合配图，包含清晰的场景描述

故事内容："""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "你是一位富有创意的故事作家。"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.8,
            max_tokens=1000
        )
        return response.choices[0].message.content.strip()
    except Exception:
        # Do NOT print the exception itself (may contain non-ASCII)
        print("[Error] Story generation failed. Check token and model.")
        return None

def extract_scenes(story, num_scenes=3):
    prompt = f"""从以下故事中提取{num_scenes}个最适合配图的关键场景，每句描述画面感强：

故事内容：
{story}

格式：
场景1：xxx
场景2：xxx
场景3：xxx"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "你是场景分析师。"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=300
        )
        lines = response.choices[0].message.content.strip().split("\n")
        scenes = []
        for line in lines:
            if "场景" in line:
                if "：" in line:
                    scenes.append(line.split("：", 1)[1].strip())
                elif ":" in line:
                    scenes.append(line.split(":", 1)[1].strip())
        return scenes[:num_scenes]
    except Exception:
        print("[Error] Scene extraction failed.")
        return []

# ---------- Image Generation (ERNIE-ViLG) ----------
def generate_image(prompt, output_path=None):
    if output_path is None:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(OUTPUT_DIR, f"image_{ts}.png")
    try:
        import erniebot
        erniebot.api_type = "aistudio"
        erniebot.access_token = ACCESS_TOKEN
        enhanced_prompt = f"{prompt}, 高质量, 精美细节, 艺术风格"
        w, h = map(int, IMAGE_SIZE.split('x'))
        response = erniebot.Image.create(
            model=IMAGE_MODEL,
            prompt=enhanced_prompt,
            height=h,
            width=w,
            num=1
        )
        img_data = response.get_result()
        with open(output_path, "wb") as f:
            f.write(img_data)
        return output_path
    except ImportError:
        print("[Warning] erniebot not installed. Run: !pip install erniebot")
        return None
    except Exception:
        print("[Error] Image generation failed.")
        return None

def generate_scene_images(scenes):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    paths = []
    for i, scene in enumerate(scenes):
        print(f"Generating image {i+1}/{len(scenes)} ...")
        out_path = os.path.join(OUTPUT_DIR, f"scene_{i+1}.png")
        img_path = generate_image(scene, out_path)
        if img_path:
            paths.append(img_path)
    return paths

# ---------- Text-to-Speech (edge-tts) ----------
def text_to_speech(text, output_path=None):
    if output_path is None:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(OUTPUT_DIR, f"story_{ts}.mp3")
    try:
        import edge_tts
        paragraphs = [p for p in text.split('\n') if p.strip()]
        full_audio = bytearray()
        async def synth(para):
            communicate = edge_tts.Communicate(para, TTS_VOICE)
            audio = b""
            async for chunk in communicate.stream():
                if chunk["type"] == "audio":
                    audio += chunk["data"]
            return audio
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        for para in paragraphs:
            audio_chunk = loop.run_until_complete(synth(para))
            full_audio.extend(audio_chunk)
        loop.close()
        with open(output_path, "wb") as f:
            f.write(full_audio)
        return output_path
    except ImportError:
        print("[Warning] edge-tts not installed. Run: !pip install edge-tts")
        return None
    except Exception:
        print("[Error] TTS failed.")
        return None

# ---------- Main ----------
def main():
    print("=" * 60)
    print("AI Creative Story Studio (Paddle AI Studio)")
    print("=" * 60)

    topic = input("\nPlease enter story theme: ").strip()
    if not topic:
        print("Theme cannot be empty!")
        return

    print("\n[Info] Generating story... (about 10-20s)")
    story = generate_story(topic)
    if not story:
        print("[Error] Story generation failed. Check token and model availability.")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    story_path = os.path.join(OUTPUT_DIR, "story.txt")
    with open(story_path, "w", encoding="utf-8") as f:
        f.write(story)

    print(f"\n[Success] Story saved to: {story_path}")
    print("\n" + "=" * 60)
    print("Generated Story (Chinese):")
    print("=" * 60)
    print(story)
    print("=" * 60)

    print("\n[Info] Extracting scenes...")
    scenes = extract_scenes(story)
    if scenes:
        print(f"[Success] Extracted {len(scenes)} scenes")
        print("\n[Info] Generating images (15-30s each)...")
        image_paths = generate_scene_images(scenes)
        for i, path in enumerate(image_paths):
            print(f"   Scene {i+1} image: {path}")
    else:
        print("[Warning] Scene extraction failed, skipping images.")

    print("\n[Info] Generating speech...")
    audio_path = text_to_speech(story)
    if audio_path:
        print(f"[Success] Audio saved to: {audio_path}")
    else:
        print("[Warning] TTS failed, skipping.")

    print("\n" + "=" * 60)
    print("All done! Files are in 'outputs/' directory.")
    print("=" * 60)

if __name__ == "__main__":
    main()

AI Creative Story Studio (Paddle AI Studio)



Please enter story theme:  一只会拉小提琴的章鱼在深海的城市里举办音乐会



[Info] Generating story... (about 10-20s)


[2026-06-24 10:49:57,041] [    INFO] _client.py:1025 - HTTP Request: POST https://aistudio.baidu.com/llm/lmapi/v3/chat/completions "HTTP/1.1 200 OK"



[Success] Story saved to: outputs/story.txt

Generated Story (Chinese):
**《深海乐章》**  

在幽蓝的深海峡谷中，隐藏着一座由珊瑚与贝壳筑成的城市——琉音城。这里的居民是各种会演奏乐器的海洋生物，而最负盛名的，是一只名叫奥克塔维奥的紫色章鱼。  

奥克塔维奥有八条灵巧的触手，其中两条专门用来握持一把特制的小提琴。这把琴由珍珠母贝打造，琴弦是坚韧的海藻丝。每当夜幕降临，荧光水母点亮城市，奥克塔维奥就会登上中央广场的贝壳舞台，开始他的演奏。  

今晚是琉音城一年一度的“深渊音乐会”。观众席上坐满了海马、海龟、发光鱼群，甚至还有几条威严的鲸鱼悬浮在远处。奥克塔维奥深吸一口海水，触手轻扬，琴弓划过琴弦。第一个音符如涟漪般荡开，整个海洋仿佛静止了一秒。  

他的音乐时而如暗流汹涌，时而似细沙轻拂。一条年轻的座头鲸忍不住跟着哼唱，声音低沉悠远；一群小丑鱼组成合唱团，用清脆的嗓音应和。最神奇的是，随着旋律起伏，周围的海水开始闪烁微光，仿佛音符化作了实体。  

演奏进入高潮时，奥克塔维奥的触手几乎舞成幻影。突然，他停下琴弓，用一条触手轻轻敲击琴身——一段即兴的打击乐响起，观众们惊喜地拍打鳍肢或摇晃贝壳，整座城市变成了巨大的交响乐团。  

曲终时，奥克塔维奥优雅地鞠躬，荧光水母们将光芒调至最亮，宛如繁星坠落深海。座头鲸发出长鸣致谢，而小章鱼们早已立志要成为下一个“八爪音乐家”。  

从此，每当洋流经过琉音城，都会捎走一缕未散的音乐，让整片海洋都知道——这里住着一只会拉小提琴的章鱼，和他的深海乐章。  

（字数：498）  

**配图建议**：  
1. 全景：荧光闪烁的珊瑚城市，中央贝壳舞台上的紫色章鱼正在演奏，周围环绕着鱼群观众。  
2. 特写：奥克塔维奥的触手灵活操控小提琴，珍珠琴身与海藻琴弦细节精致。  
3. 动态场景：音乐化作可见的音符波纹，带动海水中的发光浮游生物随之舞动。

[Info] Extracting scenes...


[2026-06-24 10:50:08,294] [    INFO] _client.py:1025 - HTTP Request: POST https://aistudio.baidu.com/llm/lmapi/v3/chat/completions "HTTP/1.1 200 OK"


[Success] Extracted 3 scenes

[Info] Generating images (15-30s each)...
Generating image 1/3 ...
[Warning] erniebot not installed. Run: !pip install erniebot
Generating image 2/3 ...
[Warning] erniebot not installed. Run: !pip install erniebot
Generating image 3/3 ...
[Warning] erniebot not installed. Run: !pip install erniebot

[Info] Generating speech...
[Warning] edge-tts not installed. Run: !pip install edge-tts
[Warning] TTS failed, skipping.

All done! Files are in 'outputs/' directory.
